In [5]:
import fitz
import pandas as pd
import sys
from pathlib import Path
from ttrpg_ocr.classify import classify_pages
from ttrpg_ocr.classify import classify_pages
from ttrpg_ocr.schemas import BookProfile
import yaml


In [2]:

PDF = Path("/Users/arturo/dev/ttrpg-ocr/data/raw/WFRP - Marienburg Sold Down The River.pdf")
OUT = Path("/Users/arturo/dev/ttrpg-ocr/data/raw/samples"); OUT.mkdir(exist_ok=True, parents=True)

doc = fitz.open(PDF)
print(f"Pages: {len(doc)}")


for i in [0, 5, 50, 80, 150, 280]:
    if i < len(doc):
        page = doc[i]
        print(f"\nPage {i}:")
        print(f"Page {i} embedded text: {len(doc[i].get_text().strip())} chars")
        # Physical page size (in points; 72 points = 1 inch)
        print(f"Page size: {page.rect.width} × {page.rect.height} pts")
        print(f"          = {page.rect.width/72:.2f} × {page.rect.height/72:.2f} inches")

        # Embedded images on the page and their actual pixel dimensions
        for img in page.get_images(full=True):
            xref = img[0]
            info = doc.extract_image(xref)
            print(f"  Image: {info['width']} × {info['height']} px, format {info['ext']}")

        pix = doc[i].get_pixmap(dpi=300)
        pix.save(OUT / f"p{i:03d}.png")

Pages: 168

Page 0:
Page 0 embedded text: 0 chars
Page size: 604.7999877929688 × 784.5599975585938 pts
          = 8.40 × 10.90 inches
  Image: 1260 × 1635 px, format jpeg

Page 5:
Page 5 embedded text: 2786 chars
Page size: 592.0 × 786.0 pts
          = 8.22 × 10.92 inches
  Image: 1124 × 111 px, format jpeg
  Image: 87 × 117 px, format jpeg
  Image: 99 × 123 px, format jpeg
  Image: 88 × 117 px, format jpeg
  Image: 90 × 118 px, format jpeg
  Image: 94 × 118 px, format jpeg

Page 50:
Page 50 embedded text: 2 chars
Page size: 595.0 × 788.0 pts
          = 8.26 × 10.94 inches
  Image: 1135 × 119 px, format jpeg
  Image: 1054 × 1391 px, format jpeg

Page 80:
Page 80 embedded text: 4236 chars
Page size: 591.0 × 785.0 pts
          = 8.21 × 10.90 inches
  Image: 308 × 13 px, format jpeg
  Image: 24 × 360 px, format jpeg
  Image: 15 × 358 px, format jpeg
  Image: 310 × 12 px, format jpeg
  Image: 1139 × 112 px, format jpeg
  Image: 504 × 381 px, format jpeg
  Image: 905 × 199 px, format jp

In [11]:
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

pdf_path = repo_root / "data" / "raw" / "WFRP - Marienburg Sold Down The River.pdf"
profile_path = repo_root / "config" / "marienburg.yaml"

profile = BookProfile.model_validate(yaml.safe_load(profile_path.read_text()))

decisions = classify_pages(pdf_path, profile)
print(len(decisions), "pages classified")

df = pd.DataFrame([d.model_dump() for d in decisions])
print(df.head())

168 pages classified
   page_num                  strategy               reason  text_chars  \
0         0         PageStrategy.SKIP        explicit drop           0   
1         1         PageStrategy.SKIP        explicit drop         352   
2         2         PageStrategy.SKIP        explicit drop        2556   
3         3  PageStrategy.NATIVE_TEXT  native text present        6044   
4         4  PageStrategy.NATIVE_TEXT  native text present        1983   

   image_count  
0            1  
1           10  
2           10  
3            2  
4            7  


In [12]:
# show all rows in a printable form
print(df.to_string(index=False))

# summary counts by classification
print(df["strategy"].value_counts())
print(df["reason"].value_counts())

# inspect the key columns only
print(df[["page_num", "strategy", "reason", "text_chars", "image_count"]])

 page_num                 strategy                       reason  text_chars  image_count
        0        PageStrategy.SKIP                explicit drop           0            1
        1        PageStrategy.SKIP                explicit drop         352           10
        2        PageStrategy.SKIP                explicit drop        2556           10
        3 PageStrategy.NATIVE_TEXT          native text present        6044            2
        4 PageStrategy.NATIVE_TEXT          native text present        1983            7
        5 PageStrategy.NATIVE_TEXT          native text present        2786            6
        6 PageStrategy.NATIVE_TEXT          native text present        1711            3
        7 PageStrategy.NATIVE_TEXT          native text present        5523            2
        8 PageStrategy.NATIVE_TEXT          native text present        5546            2
        9 PageStrategy.NATIVE_TEXT          native text present        5274            2
       10 PageStrateg